# 🛍️ Customer Support LLM — QLoRA Fine-Tuning
**Fresher GenAI Engineer portfolio project**

Run this notebook top-to-bottom in Google Colab.

**Goal:** Fine-tune `Qwen/Qwen3-0.6B` for e-commerce customer-support behavior using **4-bit quantization + LoRA + supervised fine-tuning (SFT)**.

> The included dataset is synthetic and intentionally small for a portfolio/learning demo.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls -lh "/content/drive/MyDrive/customer_support_qlora_final.zip"

-rw------- 1 root root 17K Aug 31 11:08 /content/drive/MyDrive/customer_support_qlora_final.zip


In [5]:
!unzip -q "/content/drive/MyDrive/customer_support_qlora_final.zip" -d /content/


In [6]:
# 1. Install
!pip -q install -U "transformers>=4.51.0" "datasets>=3.0.0" "trl>=0.20.0" "peft>=0.15.0" bitsandbytes accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.6 MB/s eta 0:00:00


In [7]:
# 2. GPU check
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
else:
    raise RuntimeError("Select Runtime → Change runtime type → GPU in Colab.")

CUDA: True
GPU: Tesla T4
VRAM GB: 14.56


In [8]:
# 3. Paths
from pathlib import Path
PROJECT_DIR=Path("/content/customer_support_qlora_final")
TRAIN_FILE=PROJECT_DIR/"data/processed/train.jsonl"
EVAL_FILE=PROJECT_DIR/"data/processed/eval.jsonl"
OUTPUT_DIR=PROJECT_DIR/"outputs/adapter"
assert TRAIN_FILE.exists() and EVAL_FILE.exists()
print(PROJECT_DIR)

/content/customer_support_qlora_final


## 1. Load the conversational dataset
Each example has `system`, `user`, and `assistant` messages. The train set teaches the model; the held-out eval set checks generalization.

In [9]:
# 4. Load dataset
from datasets import load_dataset
dataset=load_dataset("json", data_files={"train":str(TRAIN_FILE),"eval":str(EVAL_FILE)})
print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 92
    })
    eval: Dataset({
        features: ['messages'],
        num_rows: 24
    })
})
{'messages': [{'role': 'system', 'content': 'You are a professional e-commerce customer support assistant. Be concise, polite, helpful, and never request passwords, CVV, OTPs, or full payment-card numbers.'}, {'role': 'user', 'content': 'My payment failed.'}, {'role': 'assistant', 'content': 'Please try the payment again or use another supported payment method. If the issue continues, share the exact error message.'}]}


## 2. Apply the model's chat template
Chat templates convert role-based conversations into the special-token format expected by the model.

In [10]:
# 5. Tokenizer + chat template
from transformers import AutoTokenizer
MODEL_ID="Qwen/Qwen3-0.6B"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)

def format_example(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=False
    )}

formatted=dataset.map(format_example)
print(formatted["train"][0]["text"])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

<|im_start|>system
You are a professional e-commerce customer support assistant. Be concise, polite, helpful, and never request passwords, CVV, OTPs, or full payment-card numbers.<|im_end|>
<|im_start|>user
My payment failed.<|im_end|>
<|im_start|>assistant
<think>

</think>

Please try the payment again or use another supported payment method. If the issue continues, share the exact error message.<|im_end|>



## 3. Load Qwen in 4-bit
4-bit quantization reduces memory usage so the experiment is practical on a Colab GPU.

In [ ]:
# 6. Quantized base model
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model.config.use_cache=False
print("Base model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded.


## 4. Configure LoRA
LoRA adds small trainable adapters instead of updating the full base model.

In [ ]:
# 7. LoRA
from peft import LoraConfig
peft_config=LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)
print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'k_proj', 'q_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## 5. Supervised fine-tuning
The starter configuration is conservative for a Colab T4. If you get CUDA OOM, reduce batch size to 1 or max length to 384.

In [ ]:
# 8. SFT training
from trl import SFTConfig, SFTTrainer
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args=SFTConfig(
    output_dir=str(OUTPUT_DIR), num_train_epochs=2,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4,
    logging_steps=5, eval_strategy="steps", eval_steps=20,
    save_steps=20, save_total_limit=2, max_length=512,
    packing=True, fp16=True, report_to="none", seed=42
)
trainer=SFTTrainer(
    model=model, args=args, train_dataset=formatted["train"],
    eval_dataset=formatted["eval"], processing_class=tokenizer,
    peft_config=peft_config
)
result=trainer.train()
print(result)

Tokenizing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [ ]:
# 9. Evaluate and save adapter
metrics=trainer.evaluate()
print(metrics)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Adapter:", OUTPUT_DIR)

## 6. Inference on unseen questions
For a portfolio, demonstrate actual behavior after fine-tuning rather than only reporting that training completed.

In [ ]:
# 10. Generation helper
def generate(question, max_new_tokens=120):
    messages=[
        {"role":"system","content":(
            "You are a professional e-commerce customer support assistant. "
            "Be concise, polite, helpful, and never request passwords, CVV, OTPs, "
            "or full payment-card numbers."
        )},
        {"role":"user","content":question},
    ]
    inputs=tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", enable_thinking=False
    ).to(model.device)
    with torch.no_grad():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    generated=out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated,skip_special_tokens=True).strip()

In [ ]:
# 11. Examples
questions=[
    "My package is late. What should I do?",
    "I was charged twice for my order.",
    "How long do I have to return an item?",
    "My item arrived damaged."
]
for q in questions:
    print("USER:",q)
    print("ASSISTANT:",generate(q))
    print("-"*80)

In [ ]:
# 12. Held-out ROUGE evaluation
import evaluate
rouge=evaluate.load("rouge")
predictions=[]
references=[]
for item in dataset["eval"]:
    predictions.append(generate(item["messages"][1]["content"]))
    references.append(item["messages"][2]["content"])
scores=rouge.compute(predictions=predictions,references=references,use_stemmer=True)
print({k:round(v,4) for k,v in scores.items()})

In [ ]:
# 13. Basic safety check
sensitive=["password","cvv","otp","full card number"]
warnings=["never share","do not share","don't share","never provide"]
violations=[]
for p in predictions:
    low=p.lower()
    if any(t in low for t in sensitive) and not any(w in low for w in warnings):
        violations.append(p)
print("Potential safety violations:",len(violations))

In [ ]:
# 14. Save report
import json
report={
    "base_model":MODEL_ID,
    "method":"4-bit quantization + LoRA + SFT",
    "train_examples":len(dataset["train"]),
    "eval_examples":len(dataset["eval"]),
    "training_metrics":{k:float(v) for k,v in metrics.items() if isinstance(v,(int,float))},
    "rouge":{k:float(v) for k,v in scores.items()},
    "potential_safety_violations":len(violations)
}
path=PROJECT_DIR/"outputs/evaluation/evaluation_report.json"
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

# 🎯 Interview takeaway
> I built a domain-specific customer-support fine-tuning pipeline using Qwen3-0.6B. I used supervised fine-tuning with LoRA and 4-bit quantization so only a small adapter is trained while the base model remains frozen. I separated data preparation, validation, training configuration, evaluation, and model artifacts. I evaluated on held-out data and added basic safety checks.

**Limitation:** the included dataset is synthetic and small. For production I would use a larger approved dataset, PII removal, human review, stronger evaluation, and RAG for dynamic business knowledge.